# L15: 预训练语言模型


# L15: 预训练语言模型

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 解释预训练-微调范式的核心思想与优势
2. 比较Word2Vec、ELMo、GPT、BERT等预训练模型的技术路线
3. 理解BERT的掩码语言模型（MLM）和下一句预测（NSP）预训练任务
4. 掌握使用HuggingFace Transformers加载预训练模型并进行微调
5. 分析不同预训练模型的特点与适用场景

## 2. 核心知识点

### 2.1 预训练范式概述

**为什么需要预训练**:
- 从头训练需要大量标注数据和计算资源
- 预训练模型学习到的通用知识可以迁移到下游任务
- 预训练+微调比从零开始更有效

**预训练方法对比**:
| 方法 | 训练方式 | 典型模型 | 特点 |
|------|----------|----------|------|
| Word2Vec | 无监督 | CBOW, Skip-gram | 词级别、静态向量 |
| ELMo | 无监督 | biLSTM | 上下文相关、双向 |
| GPT | 自回归 | GPT, GPT-2 | 单向、生成能力强 |
| BERT | 掩码预测 | BERT, RoBERTa | 双向、理解能力强 |
| T5 | Seq2Seq | T5, FLAN-T5 | 统一框架、多任务 |

### 2.2 Word2Vec回顾


In [ ]:
from gensim.models import Word2Vec

# 训练Word2Vec
sentences = [
    ['machine', 'learning', 'is', 'a', 'subset', 'of', 'ai'],
    ['deep', 'learning', 'uses', 'neural', 'networks'],
    ['natural', 'language', 'processing', 'is', 'fascinating'],
    ['machine', 'learning', 'algorithms', 'learn', 'from', 'data'],
]

model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, epochs=100)

# 获取词向量
print("词向量示例:")
print(f"  'machine' 向量维度: {model.wv['machine'].shape}")
print(f"  'learning' 向量维度: {model.wv['learning'].shape}")

# 相似词
print(f"\n与'machine'最相似的词: {model.wv.most_similar('machine', topn=3)}")


### 2.3 BERT架构与预训练任务

**BERT的核心创新**:
- 双向上下文建模（同时看左边和右边）
- 掩码语言模型（Masked Language Model）
- 下一句预测（Next Sentence Prediction）

**MLM任务**:


In [ ]:
输入：[CLS] The man went to [MASK] store [SEP]
目标：[MASK] = "new"


**NSP任务**:


In [ ]:
输入：[CLS] I like cats [SEP] They are cute [SEP]  → IsNext
输入：[CLS] I like cats [SEP] The sky is blue [SEP]  → NotNext


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# 加载预训练BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 示例文本
text = "[CLS] BERT is a revolutionary model for NLP [SEP]"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

# 前向传播
input_ids = torch.tensor([token_ids])
outputs = model(input_ids)

# 获取[CLS]向量（可用于分类）
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(f"输入token: {tokens}")
print(f"[CLS]向量维度: {cls_embedding.shape}")


### 2.4 GPT系列与自回归语言模型

**GPT vs BERT**:
| 方面 | GPT (自回归) | BERT (双向) |
|------|--------------|-------------|
| 注意力方向 | 单向（只看左上文） | 双向（看全文） |
| 预训练任务 | 下一个token预测 | MLM + NSP |
| 优势 | 生成任务、对话 | 理解任务、分类 |
| 代表模型 | GPT-2/3/4 | BERT, RoBERTa |

**GPT生成示例**:


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

prompt = "Artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

# 生成
output = model.generate(
    input_ids,
    max_length=50,
    num_beams=5,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    do_sample=True
)

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"生成结果: {generated_text}")


### 2.5 HuggingFace Transformers使用


In [ ]:
"""
L15-huggingface.py
HuggingFace Transformers完整使用流程
"""

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

# ============ 1. 使用Pipeline（最简单方式） ============
print("=" * 50)
print("Pipeline方式（最简单）")
print("=" * 50)

# 情感分析
sentiment_pipe = pipeline("sentiment-analysis")
result = sentiment_pipe("I love this product! It's amazing!")
print(f"情感分析: {result}")

# 问答
qa_pipe = pipeline("question-answering")
context = """
Hugging Face is a company that develops tools for natural language processing.
They created the Transformers library which is widely used in the AI community.
"""
question = "What did Hugging Face create?"
result = qa_pipe(question=question, context=context)
print(f"问答: {result}")

# 文本生成
text_gen = pipeline("text-generation", model="gpt2")
prompt = "Once upon a time in a distant land,"
result = text_gen(prompt, max_length=50, num_return_sequences=2)
print(f"文本生成: {result[0]['generated_text']}")

# ============ 2. 加载模型和分词器 ============
print("\n" + "=" * 50)
print("手动加载模型")
print("=" * 50)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 分词
text = "This is an example sentence for tokenization."
tokens = tokenizer(text, padding=True, truncation=True, max_length=128, return_tensors='pt')
print(f"分词结果: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")
print(f"Attention mask: {tokens['attention_mask']}")

# 前向传播
outputs = model(**tokens)
print(f"logits: {outputs.logits}")


## 3. 代码示例


In [ ]:
"""
L15-fine-tuning.py
完整微调流程：文本分类任务
"""

from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import torch

# ============ 1. 加载数据和模型 ============
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 加载IMDB数据集
dataset = load_dataset("imdb")
print(f"数据集大小: {len(dataset['train'])}")

# ============ 2. 数据预处理 ============
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 划分训练/验证集
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(1000))  # 快速演示用少量数据
eval_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(500))

# ============ 3. 训练配置 ============
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# ============ 4. 评估指标 ============
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

# ============ 5. 训练 ============
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# ============ 6. 评估 ============
results = trainer.evaluate()
print(f"\n评估结果: {results}")

# ============ 7. 推理 ============
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()
    return "正面" if prediction == 1 else "负面"

print(f"\n推理测试:")
print(f"  '这部电影太棒了！' → {predict('这部电影太棒了！')}")
print(f"  '非常失望，浪费时间' → {predict('非常失望，浪费时间')}")


## 4. 练习题

1. **对比实验**: 比较Word2Vec、GPT-2、BERT三种预训练模型在词语相似度任务上的表现，分析原因。
2. **BERT预训练理解**: 解释BERT的MLM任务为什么使用"[MASK]"而不是直接删除词，这带来了什么问题？
3. **模型选择**: 对于以下任务，选择最合适的预训练模型并说明理由：文本分类、机器翻译、文本生成、问答系统。
4. **微调实验**: 在GLUE基准的一个任务上，使用BERT进行微调实验，比较不同学习率、不同训练轮次对结果的影响。
5. **知识蒸馏**: 设计一个知识蒸馏方案，将大模型（如BERT-large）的知识迁移到小模型（如BERT-small），评估压缩率和性能损失。

## 5. 延伸阅读

- **论文**: Devlin et al., 2018, "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding"
- **论文**: Radford et al., 2019, "Language Models are Unsupervised Multitask Learners" — GPT-2
- **网站**: HuggingFace官方文档 — https://huggingface.co/docs/transformers/
- **课程**: fast.ai NLP课程 — https://www.fast.ai/
- **工具**: PEFT (Parameter Efficient Fine-Tuning) — https://github.com/huggingface/peft

---
